# GEOG 592 — GeoPandas Practice Sheet

In [ ]:
import geopandas as gpd

# Load your data here, e.g.:
# parks = gpd.read_file("parks.shp")
# stops = gpd.read_file("stops.shp")
# urban = gpd.read_file("urban.shp")

## 1. Complete the hierarchy:
1. Shapely geometry
2. GeoSeries
3. GeoPandas

## 2. What type of object is returned by each expression?
Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series.

| Expression | Type |
|---|---|
| `parks.geometry` | GeoSeries |
| `parks.geometry.iloc[0]` |  Shapely geometry |
| `parks.iloc[0]` | Pandas Series |
| `parks.iloc[[0]]` | GeoDataFrame |

## 3. Explain in one sentence why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry`.

**Answer:** iloc[0] pulls only [0] parks.geogmetry pulls the whole GeoSeries of all park geometries

## 4. Write code to display all column names in `parks` as a regular Python list.

In [ ]:
list(parks.columns)

## 5. Write code to select only the `NAME` and `geometry` columns from `parks`.

In [ ]:
print(parks[["NAME","geometry"]])

## 6. Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame.

In [ ]:
parks.head(3)

## 7. Write code to inspect the geometry type of every feature in `parks`.

In [ ]:
parks.geom_type

## 8. What does `parks.crs` tell you?

**Answer:** How the coordiantes of the park relate to physical locations on Earth

## 9. Why can one point still produce a complete graph when you call `.plot()`?

**Answer:** Because .plot looks at the max and min of data sets

## 10. Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in `one_park`?

**Answer:** shapely

In [ ]:
stops.crs
# EPSG:4326

## 11. Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly.

**Answer:** No, EPSG 4326 measures only in degrees, would have to reproject into a projection that measures in ft.

## 12. Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264.

In [ ]:
stops_nc = stops.to_crs(epsg=6543)

## 13. Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area.

In [ ]:
  parks_nc = parks.to_crs(epsg=6543)
  parks_nc["area_sqft"] = parks_nc.geometry.area

## 14. Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`.

In [ ]:
parks_nc = parks.to_crs(epsg=6543)
stops_nc = stops.to_crs(epsg=6543)
park_1 = parks_2264.geometry.iloc[0]
stops_nc["distance_to_park1"] = stops_nc.geometry.distance(park_1) 

stops_nc.sort_values("distance_to_park1")[["NAME", "distance_to_park1"]]

## 15. If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain.

**Answer:** .distance(park_1) returns the shortest distance to the parks' nearest edge, not to its centriod, it equals 0 if the geometries touch or overlap

## 16. Explain each

| Predicate | Meaning |
|---|---|
| within | A lies completely inside B, with no part of A outside B |
| contains | A completely encolses B. Opposite of within|
| intersects | A and B shre at least one point, whether in the interior or on the boundary |
| touches | A and B share boundary points only, interiors don't overlap|

## 17. In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which?

**Answer:** Each geomentry in parks is tested to see if it falls with in urban.

## 18. A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True?

**Answer:** within would be false because part of the park sits oustide the urban area. intersects would be True, because the two share area

## 19. Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons.

In [ ]:
joined = gpd.sjoin(stops, parks, predicate="within")
counts = joined.groupby("index_right").size().reindex(parks.index, fill_value=0)

In [ ]:
urban_parks = gpd.sjoin(parks, urban, predicate="within")

## 20. What is the purpose of `gpd.sjoin()`?

**Answer:** Joins two GeoDataFrames by their spatial relationship (within, intersects, contains) instead of by a shared column. The result is attaching attributes from the right GeoDataFrame to the rows of the left one where their geometries meet the pedicate.

## 21. Why is a spatial join different from a regular pandas join on a common ID field?

**Answer:** Pandas join - matches equal values in a shared ID column. 
Spatial join - matches rows where their geometries sit relative to each other. A single row can match several rows or none

## 22. Write a spatial join that keeps bus stops that fall inside park polygons.
Use `stops` as the left GeoDataFrame and `parks` as the right GeoDataFrame.

In [ ]:
stops_in_parks = gpd.sjoin(stops, parks, predicate="within")

## 23. After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code.

**Answer:** 

In [ ]:
stops_in_parks = gpd.sjoin(stops, parks, predicate="within")
counts = stops_in_parks.groupby("index_right").size().reindex(parks.index, fill_value=0)

## 24. Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size?

**Answer:** Each match between a left and right polygon becomes its own output row. One polygon can overlap or touch more than one polygon. 

## 25. You want to answer: “Which parks are within 500 feet of a bus stop?” Write how you would do this with GeoPandas

### a. Using a distance and apply method

In [ ]:
stops_nc = stops.to_crs(epsg=6543)
parks_nc = parks.to_crs(epsg=6543)

all_stops = stops.geometry.union.all()
near_parks = parks[parks.geometry.distance(all_stops) < 500]

### b. Using buffers

In [9]:
stops_nc = stops.to_crs(epsg=6543)
parks_nc = parks.to_crs(epsg=6543)

stop_zones = stops_nc.copy()
stop_zones["geometry"] = stop_zones["geometry"].buffer(500)
near_parks = gpd.sjoin(parks_nc, stop_zones, predicate="intersects")
near_parks = near_parks.drop_duplicates(subset=parks.columns.drop("geometry"))

NameError: name 'stops' is not defined